# Exploring i.i.d. mutation effects

The effects of mutation on the growth rate (fitness) of offspring are independently and identically distributed at $L=2^n$ loci.

In [1]:
%matplotlib notebook
%run bs.py
sns.set_style("darkgrid", {"axes.facecolor": ".9"})
sns.set_context("notebook", font_scale=1, rc={"lines.linewidth": 2})

The effects of mutation on the growth rate of the offspring range over the interval&hellip;

In [2]:
[-MAX_BIRTH_RATE, MAX_BIRTH_RATE]

[-0.25, 0.25]

## Gaussian distribution of mutation effects

We start by considering a discretized $\text{Normal}(0,\, 0.05)$ distribution over mutation effects. Rather than suppress the processing of the given distribution by giving `None` as the value for `number_of_mutations` and/or `log_number_of_loci`, we provide the values for which the processing should have no effect on the distribution.

In [3]:
effects = GaussianEffects(n_rates=1001, std=0.05, number_of_mutations=1, log_number_of_loci=0)

### Some validation

The parameter `n_rates=1001` gives the number of growth rates. The number of possible changes in growth rate due to a single mutation is `2 * n_rates - 1`.

In [4]:
len(effects)

2001

Check to see that the endpoints of the domain of the distribution are as expected.

In [5]:
[effects.domain[0], effects.domain[-1]]

[-0.25, 0.25]

The elements of the domain should sum to zero.

In [6]:
np.sum(effects.domain)

0.0

Check to see that the distribution has been normalized.

In [7]:
np.sum(effects)  # effects[i] is equivalent to effects.p[i]

1.0

Verify that the mean and the standard deviation of the mutation effects are close to those specified. We expect the standard deviation to be somewhat less than specified because the tails of the distribution have been cropped.

In [8]:
mean, variance = effects.mean_and_variance()
mean, variance ** 0.5

(0.0, 0.049999684835421214)

Now we explicitly construct the distribution, and check to see that it matches `effects`.

In [9]:
x = np.linspace(0, MAX_BIRTH_RATE, 1001) # non-negative part of the domain
x = np.concatenate((-x[::-1], x[1:]))    # ensure perfect symmetry of domain
(x == effects.domain).all()              # do all domain elements match?

True

Note that `-x[::-1]` is the negated reverse of `x`, and that `x[1:]` excludes the first element of `x`.

In [10]:
p = Distribution.masses(stats.norm(0, 0.05), x)
p /= np.sum(p)          # normalize distribution
p = (p + p[::-1]) / 2   # ensure perfect symmetry of distribution
(p == effects.p).all()  # do all probabilities match?

TypeError: masses() missing 1 required positional argument: 'domain'

Here is a plot of the distribution.

In [ ]:
fig = plt.figure() # create a figure
axes = fig.gca()   # get current axes
effects.plot(axes, label='Normal(0, 0.05)')
axes.legend(loc='best')
axes.set_title('Mutation effects distribution, prior to self-convolution');

### $L$-fold convolution

`GaussianEffects` is a subclass of the class `EffectsDistribution`. In order to obtain the $L$-fold convolution of an `EffectsDistribution` instance, we set the number of mutations per offspring to $L = 2^n$ and the log-number of loci to $n.$ In theory, the standard deviation of the result should be $n$ times the standard deviation of the original distribution.

The following function plots&hellip;???. We immediately check to see that it reproduces the plot above when the standard devition is 0.005, the number of mutations per organism is 1, and the number of loci is 1.

In [ ]:
def plot(std, number_of_mutations, log_number_of_loci):
    fig = plt.figure()
    axes = fig.gca()
    effects = GaussianEffects(n_rates=1001, std=std, number_of_mutations=None)
    effects.plot(axes, label='Normal(0, {0:f})'.format(std))
    effects.iid_effects(number_of_mutations, log_number_of_loci, truncate_self_convolution=False)
    effects.plot(axes, label='{0}-fold convolution'.format(number_of_mutations));
    axes.legend(loc='best')
    axes.set_title('Number of mutations {0}, number of loci {1}'.format(number_of_mutations,
                                                                        2 ** log_number_of_loci))
    return effects, fig

plot(0.05, 1, 0);

There are severe edge effects in the 4-fold convolution of a zero-mean, normal distribution when the results are limited to the interval $[-5\sigma,\, 5\sigma].$

In [ ]:
plot(0.05, 2**2, 2);

So we set the standard deviation of the original distribution to $0.05/n,$ and expect the resulting distribution to approximate $\text{Normal}(0,\, 0.05).$

In [ ]:
n=4
plot(0.05/n, 2**n, n);
plt.plot(x, p, label='Normal(0, 0.05)', ls='--', lw=0.5)
plt.legend(loc='best');

Verify that the mean of the result is about zero, and that the variance is about $0.05^2 = 0.0025.$

In [ ]:
effects.mean_and_variance()

### Mutation effects

Now we consider $L = 2^n$ loci with 1 mutation per offspring. The standard deviation of the initial Gaussian distribution is equal to $0.02,$ as in Section 5.3 of the BS article. The expected number of mutations at any given site is $\mu = 1 / L.$ Thus, to obtain the distribution of probability over mutation effects at a site, we scale  the probabilities in the initial distribution by $\mu,$ and then add $1 - 1/L$ to the probability that mutation at the site has zero effect on the growth rate (fitness) of the offspring.

**TO DO**

## Symmetrized Gamma distribution of mutation effects

### Lumped effects

In [ ]:
asym = GammaEffects(percent_beneficial=0.001)  # BS Sect 5.4
sym = GammaEffects(percent_beneficial=None)

Verify that the beneficial tail of the reweighted distribution has been scaled to 0.001.

In [ ]:
mid = len(asym) // 2
beneficial = asym[mid+1:]
non_beneficial = asym[:mid+1]
s='beneficial sum {0:f} for {1} elements, non-beneficial sum {2:f} for {3} elements'
s.format(np.sum(beneficial), len(beneficial), np.sum(non_beneficial), len(non_beneficial))

Verify that the mean and variance of the (normalized) upper tail of the symmetrized Gamma distribution are close to the values $\alpha/\beta$ and $\alpha/\beta^2,$ respectively.

In [ ]:
mid = len(sym) // 2
x = sym.domain[mid:]
p = np.array(sym[mid:])
p[0] /= 2
p /= np.sum(p)
mean, variance = mean_and_variance(x, p)
s='mean, variance: {0:e}, {1:e} (expected: {2:e}, {3:e})'
s.format(mean, variance, sym.alpha/sym.beta, sym.alpha/sym.beta**2)

It is clear from the following plot why the observed mean and variance are not particularly close to the theoretically predicted values. The probability mass is concentrated on just a few discrete values. There is no reason to think that the discrete distribution would approximate the continuous distribution closely.

In [ ]:
fig = plt.figure()
axes = fig.gca()
shift = (sym.domain[1] - sym.domain[0]) / 8
plt.title('Symmetrized Gamma Before and After Reweighting')
sym.vlines(axes, -shift, label='Before').set_color(sns.xkcd_palette(['amber']))
asym.vlines(axes, shift, label='After').set_color(sns.xkcd_palette(['windows blue']))
plt.legend(loc='best')
plt.xlim(-0.01, 0.01);

Note that there are zoom and pan controls on the figure.

### Independent, identically distributed, additive mutation effects at loci

In [ ]:
log_L = 14
iid = GammaEffects(percent_beneficial=0.001, number_of_mutations=1, log_number_of_loci=log_L)
fig = plt.figure()
axes = fig.gca()
shift = (asym.domain[1] - asym.domain[0]) / 8
axes.set_title('Gamma Mutation Effects\n$L={0}$, $\mu=1/L$'.format(2**log_L))
axes.set_xlabel('Difference in Fitness of Offspring and Parent')
axes.set_ylabel('Probability')
asym.vlines(axes, -shift, label='Before').set_color(sns.xkcd_palette(['windows blue']))
iid.vlines(axes, shift, label='After').set_color(sns.xkcd_palette(['amber']))
plt.legend(loc='best')
plt.xlim(-0.01, 0.01);

**The ratio of the probability of deleterious mutation to the probability of beneficial mutation is somewhat greater with additive iid effects than with lumped effects.**

The probability of no mutation effect on growth rate is excluded from the following calculations.

In [ ]:
n = len(asym) // 2
lumped_ratio = np.sum(asym[:n]) / np.sum(asym[n+1:])
iid_ratio = np.sum(iid[:n]) / np.sum(iid[n+1:])
'Deleterious-to-beneficial ratio: lumped {0:f}, iid {1:f}'.format(lumped_ratio, iid_ratio)

## Animation
Now we animate four evolutionary processes that differ [**outdated description**] in the distribution of probability over mutation effects on the growth rates of offspring. The distributions are those of the preceding figure.

In [ ]:
n_years = 1600
init = GaussianRates(asym.n_rates)
pops = [Population(init, asym, label='One effect'),
        Population(init, iid, label='IID effects'),
        BS_Population(init, asym, label='One effect (BS)', threshold_norm=None,),
        BS_Population(init, iid, label='IID effects (BS)', threshold_norm=None),
        BS_Population(init, asym, label='One effect (BS)', threshold_norm=np.max),
        BS_Population(init, iid, label='IID effects (BS)', threshold_norm=np.max)]
processes = np.empty(len(pops), dtype=object)
processes[:] = [Evolution(p, n_years=n_years) for p in pops]
s = "\nGamma Mutation Effects, BS "
bs_infinite = CompareProcesses(processes[[0, 1, 2, 3]], subtitle=s+"Infinite")
bs_finite = CompareProcesses(processes[[0, 1, 4, 5]], subtitle=s+"Finite")

The time required to generate the following animation is proportional to the `stride` setting, which determines how many years the animator steps ahead in each frame. For a JS/HTML animation, the storage is also proportional to the stride. HTML5 comes with excellent compression, but requires that the FFmpeg package be installed on the server.

When the stride is set to None, a figure is generated instead of an animation. Figures, unlike animations, are displayed when a notebook is first loaded.

In [ ]:
plt.rcParams['animation.html'] = 'jshtml' # 'html5' requires FFmpeg
#with sns.color_palette():
#sns.set_palette("Paired")
sns.set_palette(sns.color_palette("Set2", 4))
bs_infinite.animate(stride=None)

Repeat without thresholding in the BS population.

In [ ]:
bs_finite.animate(stride=None)

## Some BS-Like Plots

In [ ]:
bs_infinite.mean_variance_plots()